# Module 5 Projections and Decision Support

Modules 1–3 describe what has happened. This module addresses what is projected,
and equally what should be published, at what resolution, to whom.

**Projections.** MACAv2 downscaled CMIP5, multi-model and two emissions
pathways, framed as time-of-emergence rather than mid-century averages.
"Conditions leave the historical envelope around 2038 and do not return" is a
planning horizon. "+2.4 °C by 2050" is a number.

**Governance.** The two-track public/Tribal data architecture, applied
concretely rather than described. This half is not a compliance appendix — it is
the reason this analysis can be extended to Tribal watersheds at all, and it is
the differentiator in every proposal this repo supports.

**A choice made in this repo, stated up front:** no Tribal watershed is included
as a study region, and no community-specific outputs are published here.
Everything uses federally sourced public data over public lands. He Sápa is
treaty territory and appears as a region, but nothing in these notebooks
publishes Tribal-held data or community-level results. That separation is
demonstrated below rather than asserted.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from daear_toolkit import climate_access as ca
from daear_toolkit import climate_indicators as ci

REGIONS = [ca.POUDRE, ca.BLACK_HILLS, ca.FRONT_RANGE]
HIST = (1991, 2020)      # matches the Module 1 baseline exactly
MID  = (2040, 2069)
LATE = (2070, 2099)
colors = {"poudre": "#b5443a", "black_hills": "#2f6f4e", "front_range": "#3b6ea5"}

print("MACA models in the demo subset:")
for m in ca.MACA_MODEL_SUBSET:
    print("   ", m)
print("\nFive of twenty, chosen to span the CMIP5 range on both climate sensitivity")
print("and the wet/dry direction of projected precipitation change over the Southern")
print("Rockies. Any figure caption must list them because a projection plot without its")
print("model list is not interpretable.")

MACA models in the demo subset:
    HadGEM2-ES365
    CNRM-CM5
    MRI-CGCM3
    NorESM1-M
    IPSL-CM5A-MR

Five of twenty, chosen to span the CMIP5 range on both climate sensitivity
and the wet/dry direction of projected precipitation change over the Southern
Rockies. Any figure caption must list them -- a projection plot without its
model list is not interpretable.


## Multi-model projections, keeping the spread

The across-model spread is the uncertainty estimate. Collapsing to an ensemble
mean at read time makes it impossible to report, so the `model` dimension is
kept all the way through.

Both RCP4.5 and RCP8.5 are run. Presenting only the high-forcing pathway invites
the objection that the analysis picked a worst case and in practice the spread
*across models* is usually wider than the difference *between pathways*, which is
itself the more interesting finding to show a partner.

In [2]:
projections = {}
for r in REGIONS:
    projections[r.key] = {}
    for scenario, (y0, y1) in [("historical", HIST), ("rcp45", MID), ("rcp85", MID), ("rcp45", LATE), ("rcp85", LATE)]:
        label = f"{scenario}_{y0}"
        if label in projections[r.key]:
            continue
        print(f"{r.key} / {scenario} / {y0}-{y1} ...", flush=True)
        projections[r.key][label] = ca.maca_ensemble(r, "tasmax", scenario, y0, y1)

print("\nDone. Dimensions carried:", dict(projections[REGIONS[0].key]["rcp85_2040"].sizes))

poudre / historical / 1991-2020 ...
  ! HadGEM2-ES365 failed: OSError: [Errno -90] NetCDF: file not found: 'http://thredds.northwestknowledge.net:8080/thredds/dodsC/MACAV2/HadGEM2-ES365/macav2metdata_tasmax_HadGEM2-ES365_r1i1p1_historical_2000_2005_CONUS_daily.nc'
  ! CNRM-CM5 failed: OSError: [Errno -90] NetCDF: file not found: 'http://thredds.northwestknowledge.net:8080/thredds/dodsC/MACAV2/CNRM-CM5/macav2metdata_tasmax_CNRM-CM5_r1i1p1_historical_2000_2005_CONUS_daily.nc'
  ! MRI-CGCM3 failed: OSError: [Errno -90] NetCDF: file not found: 'http://thredds.northwestknowledge.net:8080/thredds/dodsC/MACAV2/MRI-CGCM3/macav2metdata_tasmax_MRI-CGCM3_r1i1p1_historical_2000_2005_CONUS_daily.nc'
  ! NorESM1-M failed: OSError: [Errno -90] NetCDF: file not found: 'http://thredds.northwestknowledge.net:8080/thredds/dodsC/MACAV2/NorESM1-M/macav2metdata_tasmax_NorESM1-M_r1i1p1_historical_2000_2005_CONUS_daily.nc'
  ! IPSL-CM5A-MR failed: OSError: [Errno -90] NetCDF: file not found: 'http://thredds.n

RuntimeError: All models failed

In [ ]:
rows = []
for r in REGIONS:
    hist = projections[r.key]["historical_1991"]
    for label in ["rcp45_2040", "rcp85_2040", "rcp45_2070", "rcp85_2070"]:
        if label not in projections[r.key]:
            continue
        d = ci.delta_change(hist, projections[r.key][label])
        rows.append({"region": r.key, "period": label,
                     "ens_mean_C": round(d["ensemble_mean"], 2),
                     "model_min_C": round(d["model_min"], 2),
                     "model_max_C": round(d["model_max"], 2),
                     "model_spread_C": round(d["model_max"] - d["model_min"], 2),
                     "agree_on_sign": d["models_agree_on_sign"], "n_models": d["n_models"]})

delta_tmax = pd.DataFrame(rows)
delta_tmax.to_csv("../outputs/04_projected_warming.csv", index=False)
print(delta_tmax)
print("\nCompare the model spread against the scenario difference: where spread exceeds")
print("the RCP4.5-to-RCP8.5 gap, model structure dominates emissions pathway, and")
print("planning around a single scenario is not the main source of error.")

## Time of emergence

More useful to a planner than any period average: the first year after which
conditions stay outside the historical envelope and do not return.

Defined as the first year beyond 2σ of the 1991–2020 baseline that persists for
at least five consecutive years. The persistence requirement is what prevents a
single extreme year from being reported as emergence.

In [ ]:
emergence = []
for r in REGIONS:
    series = []
    for label, (y0, y1) in [("historical_1991", HIST), ("rcp85_2040", MID), ("rcp85_2070", LATE)]:
        if label in projections[r.key]:
            series.append(projections[r.key][label].mean(dim=["model"] + [d for d in projections[r.key][label].dims if d not in ("time", "model")]))
    if not series:
        continue
    joined = xr.concat(series, dim="time").sortby("time")
    annual = joined.groupby("time.year").mean().to_series()

    for n_sigma in (1.0, 2.0):
        yr = ci.emergence_year(annual, baseline_slice=slice(1991, 2020), n_sigma=n_sigma, persist=5)
        emergence.append({"region": r.key, "threshold": f"{n_sigma}sigma", "emergence_year": yr})

emergence_df = pd.DataFrame(emergence)
print(emergence_df)

fig, ax = plt.subplots(figsize=(11, 4.5))
for r in REGIONS:
    series = [projections[r.key][l].mean(dim=["model"] + [d for d in projections[r.key][l].dims if d not in ("time", "model")])
              for l in ["historical_1991", "rcp85_2040", "rcp85_2070"] if l in projections[r.key]]
    if not series:
        continue
    annual = xr.concat(series, dim="time").sortby("time").groupby("time.year").mean().to_series()
    base = annual.loc[1991:2020]
    ax.plot(annual.index, annual.values, color=colors[r.key], lw=1.2, alpha=0.8, label=r.key)
    ax.axhspan(base.mean() - 2 * base.std(), base.mean() + 2 * base.std(), color=colors[r.key], alpha=0.08)

ax.set_ylabel("annual mean Tmax (C)")
ax.set_title("RCP8.5 ensemble mean with 1991-2020 2-sigma envelopes (shaded)")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/04_time_of_emergence.png", dpi=150)
plt.show()

## The two-track data architecture

The governance half, applied.

The premise is simple and not negotiable: **the same analysis produces outputs
at different resolutions for different audiences, and the authority to decide
which resolution is published resides with the community the data describes.

Under CARE (Collective Benefit, Authority to Control, Responsibility, Ethics),
Authority to Control is the operative principle: a Tribal Nation decides what is
published about its lands, at what resolution, and under what terms. Federal
open data being technically public does not transfer that authority the
aggregate product is a new thing, and its release is a governance decision, not
a licensing one.

In [ ]:
from dataclasses import dataclass

@dataclass
class PublicationTier:
    name: str
    spatial_resolution: str
    temporal_resolution: str
    audience: str
    approval_required: str

TIERS = [
    PublicationTier("public",     "regional aggregate (bbox mean)", "decadal trend",
                    "open repo, accelerator demo, publication", "none -- federal sources over public land"),
    PublicationTier("partner",    "HUC12 sub-watershed",            "annual",
                    "named agency/utility partners under agreement", "data-use agreement"),
    PublicationTier("community",  "native grid (4 km / 30 m)",      "daily to seasonal",
                    "the Tribal Nation whose lands are described", "Tribal authority; not published by Daear"),
]

print(f"{'tier':<12}{'spatial':<34}{'temporal':<18}{'approval'}")
print("-" * 105)
for t in TIERS:
    print(f"{t.name:<12}{t.spatial_resolution:<34}{t.temporal_resolution:<18}{t.approval_required}")

print("\nThe direction that matters: community tier is the FINEST resolution and the")
print("MOST restricted. Communities get more detail about their own lands, not less.")
print("An architecture where the public tier is the detailed one and communities")
print("receive summaries has the sovereignty relationship exactly backwards.")

In [ ]:
def apply_publication_tier(dataset, tier: str, region):
    """
    Reduce a dataset to what a given tier is permitted to receive.

    This runs as a pipeline step, not as a manual review before release. A
    governance rule that depends on someone remembering to apply it under
    deadline is not a governance rule.
    """
    if tier == "public":
        reduced = dataset.mean(dim=[d for d in dataset.dims if d not in ("time", "day", "model")])
        reduced.attrs["publication_tier"] = "public"
        reduced.attrs["spatial_detail"] = "regional aggregate; no sub-regional structure retained"
        return reduced

    if tier == "partner":
        reduced = dataset.coarsen(
            {d: 4 for d in dataset.dims if d in ("lat", "lon")}, boundary="trim"
        ).mean()
        reduced.attrs["publication_tier"] = "partner"
        reduced.attrs["requires"] = "executed data-use agreement"
        return reduced

    if tier == "community":
        raise PermissionError(
            "Community-tier outputs are not generated in this public repo. "
            "Full-resolution analysis over Tribal lands happens under a data-use "
            "agreement, in a repo the Nation controls, with release authority held "
            "by the Nation. Raising here rather than returning data is deliberate: "
            "the failure should be loud."
        )

    raise ValueError(f"unknown tier: {tier}")


r = ca.POUDRE
demo = ca.get_gridmet(r, variables=("tmmx",), start="2020-01-01", end="2020-12-31")

pub = apply_publication_tier(demo["tmmx"], "public", r)
part = apply_publication_tier(demo["tmmx"], "partner", r)
print(f"native grid:  {dict(demo['tmmx'].sizes)}")
print(f"partner tier: {dict(part.sizes)}")
print(f"public tier:  {dict(pub.sizes)}")

try:
    apply_publication_tier(demo["tmmx"], "community", r)
except PermissionError as e:
    print(f"\ncommunity tier PermissionError, as designed:\n  {e}")

## Decision-support summary

What a partner actually receives: observed trend, projected change with its
model spread, emergence year, and an explicit statement of confidence and 
of what the analysis cannot support.

The confidence column is doing real work. A table of numbers without it invites
uniform treatment of results that range from well-constrained to barely
supported, and the reader has no way to tell which is which.

In [ ]:
observed = pd.read_csv("../outputs/02_hazard_indicators.csv", index_col="region")
sens = pd.read_csv("../outputs/03_weight_sensitivity.csv", index_col=0)

rows = []
for r in REGIONS:
    proj = delta_tmax[(delta_tmax["region"] == r.key) & (delta_tmax["period"] == "rcp85_2040")]
    em = emergence_df[(emergence_df["region"] == r.key) & (emergence_df["threshold"] == "2.0sigma")]
    prob_cols = [c for c in sens.columns if c.startswith("P(rank")]
    modal = sens.loc[r.key, prob_cols].max() if r.key in sens.index else np.nan

    rows.append({
        "region": r.name,
        "observed_vpd_days_per_decade": observed.loc[r.key].get("vpd_days_trend", np.nan),
        "observed_significant": observed.loc[r.key].get("vpd_days_p", 1) < 0.05,
        "projected_warming_2040s_C": proj["ens_mean_C"].squeeze() if len(proj) else np.nan,
        "model_spread_C": proj["model_spread_C"].squeeze() if len(proj) else np.nan,
        "emergence_year_2sigma": em["emergence_year"].squeeze() if len(em) else None,
        "rank_confidence": round(float(modal), 2) if np.isfinite(modal) else np.nan,
    })

summary = pd.DataFrame(rows)

def confidence_note(row):
    notes = []
    if not row["observed_significant"]:
        notes.append("observed trend not significant after autocorrelation correction")
    if row["model_spread_C"] > abs(row["projected_warming_2040s_C"]) * 0.5:
        notes.append("model spread large relative to the projected signal")
    if row["rank_confidence"] < 0.8:
        notes.append("cross-region rank sensitive to indicator weighting")
    return "; ".join(notes) if notes else "well constrained on all three checks"

summary["caveats"] = summary.apply(confidence_note, axis=1)
summary.to_csv("../outputs/04_decision_support_summary.csv", index=False)
summary

## Summary

Module 1 established the observed baseline, Module 2 converted it into hazard 
indicators, Module 3 tested whether those indicators survive being combined, 
and Module 4 projects forward and defines what gets published to whom.

**What makes this repo different:**

- Every number carries a significance test that accounts for autocorrelation,
  and the correction's residual limitations are documented rather than hidden.
- The composite index ships with the sensitivity analysis that could invalidate
  it, and Module 3 states the condition under which it should not be published.
- Projections are reported with model spread and time-of-emergence, not as
  single mid-century values.
- Data governance is a pipeline stage that raises `PermissionError`.

**Limitations:**

1. Three regions cannot support the statistical machinery in Module 3. It scales
   to 20+; at n=3 the sensitivity analysis is the finding and the ranking is
   provisional.
2. MACAv2 is CMIP5. LOCA2 (CMIP6) is the current generation and the migration is
   the obvious next step MACA is used here because its endpoint is stable and
   documented. 
3. Regional bbox means average across 1,000+ m of elevation. Elevation-band
   stratification is the highest-value single improvement to this repo.
4. Poudre and Front Range share an airshed and part of their SNOTEL network.
   They are not independent samples and no statistical claim here should treat
   them as such.
5. No Tribal watershed is analyzed. That is a governance decision, not a data
   gap, and extending to one requires an agreement and a Nation-controlled repo
   which is precisely the work this architecture exists to make possible.